# Sectra First Responder: Incident-Response Chatbot (UI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/df_llm_labs/ir_first_responder/ir_chatbot_ui.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  ·  **SectraTEK Lab**

When an incident happens, the team **calls Sectra**: they say what they think is going on, and
Sectra walks them through the response using the company's **IR plan and playbooks**. It drafts
the management notification (and only **sends it - simulated -** once the team approves),
recommends the exact **collection commands**, and reads back the **artifacts** the team uploads.

> This is a training **drill**: no real emails are sent. A sample company plan + playbooks and two
> sample artifacts are bundled; you can replace the playbooks and upload your own artifacts in the UI.

### 1. Install the libraries

In [ ]:
%pip install -q gradio openai

### 2. Create the folders, then write the code and the sample playbooks/evidence

In [ ]:
import os
os.makedirs('knowledge', exist_ok=True)
os.makedirs('evidence', exist_ok=True)

In [ ]:
%%writefile agentkit.py
"""
agentkit.py - the shared forensic-agent core used by the Lab 4 notebooks and chatbots.

One place defines: how to connect to a model (OpenRouter or Ollama), how to build the
synthetic Android image, the read-only tools the agent may call, the think-act-observe
loop, and a conversational wrapper for the chatbots. The chatbot .py apps import this
module; the teaching notebooks embed the same code inline so students open only one file.
"""
import json
import os
import re
import sqlite3
from pathlib import Path


# ======================================================================= model backend
def connect_model(provider="openrouter", model=None, api_key=None, base_url=None):
    """Return (client, model_id). provider is 'openrouter' or 'ollama'.

    openrouter: free hosted Qwen, needs a key from https://openrouter.ai/keys.
    ollama:     a Qwen model running locally (https://ollama.com), no key, nothing leaves the PC.
    """
    from openai import OpenAI
    provider = provider.lower()
    if provider == "ollama":
        base_url = base_url or "http://localhost:11434/v1"
        client = OpenAI(base_url=base_url, api_key="ollama")  # Ollama ignores the key
        return client, (model or "qwen3:8b")
    # default: OpenRouter
    base_url = base_url or "https://openrouter.ai/api/v1"
    api_key = api_key or os.environ.get("OPENROUTER_API_KEY", "")
    client = OpenAI(base_url=base_url, api_key=api_key)
    return client, (model or "qwen/qwen3-30b-a3b:free")


# ======================================================================= synthetic image
def build_image(root="android_image"):
    """Create a tiny, synthetic Android extraction on disk. Everything in it is fictional."""
    ROOT = Path(root)
    text_files = {
        f"{root}/build.prop":
            "# Synthetic build.prop (teaching data, fictional)\n"
            "ro.product.manufacturer=Google\nro.product.brand=google\n"
            "ro.product.model=Pixel 7a\nro.product.device=lynx\nro.product.name=lynx\n"
            "ro.build.version.release=14\nro.build.version.sdk=34\n"
            "ro.build.fingerprint=google/lynx/lynx:14/AP2A.240705.005/2024072800:user/release-keys\n",
        f"{root}/data/system/packages.list":
            "com.google.android.gm 10089 0 /data/user/0/com.google.android.gm default 0 1\n"
            "com.android.providers.contacts 10003 0 /data/user/0/com.android.providers.contacts platform 0 1\n"
            "com.android.providers.telephony 10004 0 /data/user/0/com.android.providers.telephony platform 0 1\n"
            "com.whatsapp 10137 0 /data/user/0/com.whatsapp default 0 1\n"
            "com.instagram.android 10140 0 /data/user/0/com.instagram.android default 0 1\n"
            "org.thoughtcrime.securesms 10142 0 /data/user/0/org.thoughtcrime.securesms default 0 1\n",
        f"{root}/data/system/users/0/settings_secure.xml":
            "<?xml version='1.0' encoding='utf-8' standalone='yes' ?>\n<settings version=\"241\">\n"
            "  <setting id=\"1\" name=\"android_id\" value=\"a1b2c3d4e5f60718\" package=\"android\" />\n"
            "  <setting id=\"2\" name=\"bluetooth_name\" value=\"Pixel 7a\" package=\"com.android.settings\" />\n"
            "  <setting id=\"3\" name=\"device_name\" value=\"Nadia's Pixel 7a\" package=\"com.android.settings\" />\n"
            "</settings>\n",
        f"{root}/data/misc/wifi/WifiConfigStore.xml":
            "<?xml version='1.0' encoding='utf-8' standalone='yes' ?>\n<WifiConfigStoreData>\n  <NetworkList>\n"
            "    <Network><string name=\"SSID\">&quot;HomeNet_5G&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1721295600000\" /></Network>\n"
            "    <Network><string name=\"SSID\">&quot;BrightBeans Cafe&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1720950000000\" /></Network>\n"
            "    <Network><string name=\"SSID\">&quot;Okafor-Office&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1721210400000\" /></Network>\n"
            "  </NetworkList>\n</WifiConfigStoreData>\n",
        # a JSON artifact (many Android apps store prefs/state as JSON)
        f"{root}/data/data/com.whatsapp/files/me.json":
            json.dumps({"jid": "15550143788@s.whatsapp.net", "pushname": "Nadia O",
                        "registered": True, "last_backup": "2024-07-18"}, indent=2),
    }
    for rel, content in text_files.items():
        p = Path(rel)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content)

    def make_db(rel, schema, inserts):
        p = Path(rel)
        p.parent.mkdir(parents=True, exist_ok=True)
        if p.exists():
            p.unlink()
        con = sqlite3.connect(p)
        con.executescript(schema)
        for sql, params in inserts:
            con.executemany(sql, params)
        con.commit()
        con.close()

    make_db(f"{root}/data/data/com.google.android.gm/databases/accounts.db",
            "CREATE TABLE accounts(name TEXT, type TEXT, last_sync INTEGER);",
            [("INSERT INTO accounts VALUES (?,?,?)",
              [("nadia.okafor@gmail.com", "com.google", 1721295600),
               ("+15550143788", "com.whatsapp", 1721240000)])])
    make_db(f"{root}/data/data/com.android.providers.contacts/databases/contacts2.db",
            "CREATE TABLE contacts(display_name TEXT, phone TEXT, times_contacted INTEGER);",
            [("INSERT INTO contacts VALUES (?,?,?)",
              [("Marcus Webb", "+15550142210", 31), ("Priya Shah", "+15550148890", 12),
               ("Dad", "+15550149001", 54), ("BrightBeans Cafe", "+15550140077", 3),
               ("Alina K (work)", "+15550147325", 20)])])
    make_db(f"{root}/data/data/com.android.providers.telephony/databases/mmssms.db",
            "CREATE TABLE sms(address TEXT, date INTEGER, body TEXT, type INTEGER);",
            [("INSERT INTO sms VALUES (?,?,?,?)",
              [("+15550142210", 1721200000000, "Are we still on for Friday?", 1),
               ("+15550142210", 1721200600000, "Yes, 6pm at the usual place.", 2),
               ("+15550149001", 1721240500000, "Landed safe, call you tonight.", 2),
               ("+15550148890", 1721250000000, "Can you send the invoice?", 1),
               ("+15550149001", 1721299000000, "Thanks Dad, talk soon.", 2)])])
    make_db(f"{root}/data/data/com.whatsapp/databases/msgstore.db",
            "CREATE TABLE messages(chat TEXT, timestamp INTEGER, from_me INTEGER, text TEXT);",
            [("INSERT INTO messages VALUES (?,?,?,?)",
              [("Marcus Webb", 1721260000000, 0, "Did the package arrive?"),
               ("Marcus Webb", 1721260400000, 1, "Not yet, will check tomorrow."),
               ("Priya Shah", 1721280000000, 1, "Sent the files to your email.")])])
    return ROOT


# ======================================================================= the tools
# A text-ish file is one the agent can read as text. Everything else is "binary".
_TEXT_EXT = {".txt", ".xml", ".json", ".prop", ".list", ".log", ".csv", ".ini", ".conf", ".html"}


def make_tools(root="android_image", allow_python=False):
    """Build the read-only toolbox, all sealed inside `root`. Returns {name: function}."""
    ROOT = Path(root)

    def _resolve(path):
        base = ROOT.resolve()
        path = (path or ".").strip().lstrip("/")
        target = base if path in ("", ".") else (ROOT / path).resolve()
        if target != base and base not in target.parents:
            raise ValueError("path is outside the image")
        return target

    def list_dir(path="."):
        """List what is inside a folder (folders shown with a trailing '/')."""
        t = _resolve(path)
        if not t.exists():
            return f"No such path: {path}"
        if t.is_file():
            return f"{path} is a file, not a folder. Use read_file or run_sql."
        items = [e + ("/" if (t / e).is_dir() else "") for e in sorted(os.listdir(t))]
        return "\n".join(items) if items else "(empty folder)"

    def read_file(path, max_chars=1500):
        """Read a file. Pretty-prints JSON; for SQLite databases points you to run_sql;
        for other binary files shows the type and the first bytes."""
        t = _resolve(path)
        if not t.exists() or t.is_dir():
            return f"No such file: {path}"
        head = t.read_bytes()[:16]
        if head[:15] == b"SQLite format 3":
            con = sqlite3.connect(f"file:{t}?mode=ro", uri=True)
            tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
            con.close()
            return f"[{path} is a SQLite database. Tables: {tables}. Use run_sql with a SELECT query.]"
        ext = t.suffix.lower()
        if ext not in _TEXT_EXT and b"\x00" in head:
            return (f"[{path} is a binary file ({t.stat().st_size} bytes). First bytes (hex): "
                    f"{head.hex(' ')}. It is not readable as text.]")
        text = t.read_text(errors="replace")
        if ext == ".json":
            try:
                text = json.dumps(json.loads(text), indent=2)
            except Exception:
                pass
        return text[:max_chars] + ("" if len(text) <= max_chars else "\n...[truncated]")

    def run_sql(path, query):
        """Run ONE read-only SQL SELECT against a SQLite database and return the rows."""
        t = _resolve(path)
        if not t.exists():
            return f"No such database: {path}"
        q = (query or "").strip().rstrip(";")
        if not q.lower().startswith(("select", "pragma")):
            return "Only read-only SELECT queries are allowed."
        try:
            con = sqlite3.connect(f"file:{t}?mode=ro", uri=True)
            cur = con.execute(q)
            cols = [c[0] for c in cur.description] if cur.description else []
            rows = cur.fetchmany(50)
            con.close()
        except Exception as e:
            return f"SQL error: {e}"
        if not rows:
            return "(no rows)"
        return "\n".join([" | ".join(cols)] + [" | ".join(str(c) for c in r) for r in rows])

    def find_files(name):
        """Find files anywhere in the image whose path contains `name` (case-insensitive)."""
        base = ROOT.resolve()
        needle = (name or "").lower()
        hits = []
        for p in sorted(base.rglob("*")):
            if p.is_file() and needle in str(p.relative_to(base)).lower():
                hits.append(str(p.relative_to(base)))
        if not hits:
            return f"No files matched {name!r}."
        return "\n".join(hits[:50]) + ("" if len(hits) <= 50 else f"\n...({len(hits)} total)")

    def search_text(keyword, max_hits=20):
        """Search the text inside readable files for `keyword` (case-insensitive).
        Returns path:line for each match, so you know which file to open next."""
        base = ROOT.resolve()
        needle = (keyword or "").lower()
        out = []
        for p in sorted(base.rglob("*")):
            if not p.is_file() or p.suffix.lower() not in _TEXT_EXT:
                continue
            try:
                for i, line in enumerate(p.read_text(errors="replace").splitlines(), 1):
                    if needle in line.lower():
                        out.append(f"{p.relative_to(base)}:{i}: {line.strip()[:120]}")
                        if len(out) >= max_hits:
                            return "\n".join(out) + "\n...[more matches not shown]"
            except Exception:
                continue
        return "\n".join(out) if out else f"No text matches for {keyword!r}."

    tools = {"list_dir": list_dir, "read_file": read_file, "run_sql": run_sql,
             "find_files": find_files, "search_text": search_text}

    if allow_python:
        def run_python(code):
            """ADVANCED, OPTIONAL: run a short read-only Python snippet. `ROOT` (the image folder,
            a pathlib.Path) is available. Print what you want to see. Writing to files is blocked."""
            import io
            import contextlib
            if any(bad in code for bad in ("open(", "write", "unlink", "remove", "rmtree",
                                           "os.system", "subprocess", "Popen", "__import__")):
                return "Blocked: that snippet looks like it writes or runs external commands."
            buf = io.StringIO()
            safe = {"ROOT": ROOT, "sqlite3": sqlite3, "json": json, "Path": Path, "re": re}
            try:
                with contextlib.redirect_stdout(buf):
                    exec(code, safe)  # teaching tool: sandboxed namespace, read-only guard above
            except Exception as e:
                return f"Python error: {e}\n{buf.getvalue()}"
            out = buf.getvalue().strip()
            return out[:1500] if out else "(the snippet ran but printed nothing)"
        tools["run_python"] = run_python

    return tools


def tool_descriptions(tools):
    """One line per tool, from its docstring, for the system prompt."""
    sigs = {
        "list_dir": "list_dir(path)", "read_file": "read_file(path)",
        "run_sql": "run_sql(path, query)", "find_files": "find_files(name)",
        "search_text": "search_text(keyword)", "run_python": "run_python(code)",
    }
    lines = []
    for name, fn in tools.items():
        doc = (fn.__doc__ or "").strip().split("\n")[0]
        lines.append(f"- {sigs.get(name, name)}: {doc}")
    return "\n".join(lines)


# ======================================================================= the agent
def build_system_prompt(tools):
    return (
        "You are Sectra, a digital-forensics examiner's assistant. You are investigating an Android "
        "device extraction (a copy of the phone's files) to answer the examiner's question. You cannot "
        "see any file until you open it with a tool, and you must base every answer only on what the "
        "tools return.\n\n"
        "TOOLS (use exactly one per step):\n" + tool_descriptions(tools) + "\n\n"
        "HOW TO REPLY: send ONE JSON object and nothing else.\n"
        "To use a tool:\n"
        '  {"thought": "<one short sentence>", "action": "list_dir", "action_input": {"path": "data/data"}}\n'
        "When you can answer, reply:\n"
        '  {"thought": "<one short sentence>", "final_answer": "<your answer>", "evidence_path": "<the file you used>"}\n\n'
        "RULES: investigate step by step; use ONLY what the tools return; never invent paths, tables, "
        "or data; if a tool errors, try another path or tool; if the evidence is not there, say so "
        "plainly; always cite the exact file in evidence_path. Keep thoughts to one sentence. /no_think"
    )


def parse_action(raw):
    """Pull the JSON object out of the model's reply, tolerating <think> blocks and code fences."""
    text = re.sub(r"<think>.*?</think>", "", raw, flags=re.S)
    text = text.replace("```json", "").replace("```", "")
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except Exception:
                    return None
    return None


def run_agent_iter(question, client, model, tools, system_prompt=None, max_steps=12, history=None):
    """Generator version of the agent, for live UIs. Yields a dict per step
    ({step, thought, action, args, observation}) and finally the answer
    ({final_answer, evidence_path}) or a give-up dict ({final_answer: None}).

    history: optional prior [(user, assistant)] turns, so the agent remembers a conversation.
    """
    system_prompt = system_prompt or build_system_prompt(tools)
    messages = [{"role": "system", "content": system_prompt}]
    for u, a in (history or []):
        messages.append({"role": "user", "content": u})
        messages.append({"role": "assistant", "content": a})
    messages.append({"role": "user", "content": f"Investigate and answer this question: {question}"})

    for step in range(1, max_steps + 1):
        raw = client.chat.completions.create(
            model=model, messages=messages, temperature=0).choices[0].message.content
        messages.append({"role": "assistant", "content": raw})
        move = parse_action(raw)
        if move is None:
            messages.append({"role": "user",
                             "content": "That was not valid JSON. Reply with ONE JSON object only."})
            continue
        if "final_answer" in move:
            yield {"final_answer": move["final_answer"], "evidence_path": move.get("evidence_path", "")}
            return
        tool, args = move.get("action"), move.get("action_input", {}) or {}
        if tool not in tools:
            observation = f"Unknown tool {tool!r}. Use one of: {', '.join(tools)}."
        else:
            try:
                observation = tools[tool](**args)
            except Exception as e:
                observation = f"Tool error: {e}"
        yield {"step": step, "thought": move.get("thought", ""),
               "action": tool, "args": args, "observation": observation}
        messages.append({"role": "user", "content": f"Observation:\n{observation}"})
    yield {"final_answer": None, "evidence_path": ""}


def run_agent(question, client, model, tools, system_prompt=None, max_steps=12,
              on_step=None, history=None):
    """Drive the model think->act->observe until it answers.

    on_step(dict): optional callback for live UIs (gets each step dict, then the final dict).
    history: optional prior [(user, assistant)] turns, so the agent remembers a conversation.
    Returns the final move dict (with 'final_answer'/'evidence_path'), or None if it gave up.
    """
    final = None
    for ev in run_agent_iter(question, client, model, tools, system_prompt, max_steps, history):
        if on_step:
            on_step(ev)
        if "final_answer" in ev:
            final = ev if ev["final_answer"] is not None else None
    return final


def answer_text(move):
    """Format a final move as a human answer line with its citation."""
    if not move:
        return "I could not reach an answer within the step limit. Try rephrasing the question."
    cite = move.get("evidence_path") or "(no file cited)"
    return f"{move['final_answer']}\n\nSource: {cite}"


In [ ]:
%%writefile irkit.py
"""
irkit.py - Sectra First Responder: an incident-response copilot over a company's
IR plan and playbooks.

The team "calls" Sectra, says what they think the incident is, and Sectra walks them
through the response using ONLY the company's plan and playbooks (its knowledge tools).
It can also act, with the team's approval: draft a management notification (pulling a
template from the playbook), recommend the exact collection tools/commands, and read the
artifacts the team collects back.

SAFETY / DESIGN (this is a drill):
- Side-effecting actions (sending a management email) are SIMULATED, never real. The agent
  must get the team's explicit approval in chat before it may call send_email(approved=True),
  and even then nothing is actually emailed - it is logged to an action log.
- Knowledge tools are read-only over the plan/playbooks. Evidence tools are read-only over
  whatever artifacts the team uploads. The agent cites the playbook section behind each step.

Reuses the shared agent loop from agentkit (connect_model, run_agent, run_agent_iter).
"""
import os
import re
from pathlib import Path

import agentkit as ak

connect_model = ak.connect_model
run_agent = ak.run_agent
run_agent_iter = ak.run_agent_iter
answer_text = ak.answer_text

_HEADING = re.compile(r"^(#{1,3})\s+(.*)$")
_TEXT_EXT = {".txt", ".log", ".csv", ".json", ".xml", ".md", ".ini", ".conf", ".eml"}


# ----------------------------------------------------------------- knowledge (plan + playbooks)
def load_knowledge(folder):
    """Read every .md doc in `folder` into {doc_name: [(section_title, body), ...]}."""
    knowledge = {}
    for p in sorted(Path(folder).glob("*.md")):
        knowledge[p.name] = _split_sections(p.read_text(errors="replace"))
    return knowledge


def _split_sections(text):
    sections, title, cur = [], "(top)", []
    for line in text.splitlines():
        m = _HEADING.match(line)
        if m:
            if cur:
                sections.append((title, "\n".join(cur).strip()))
            title, cur = m.group(2).strip(), []
        else:
            cur.append(line)
    sections.append((title, "\n".join(cur).strip()))
    return [(t, b) for t, b in sections if b or t != "(top)"]


# ----------------------------------------------------------------- the tools
def make_ir_tools(knowledge, evidence_root, action_log, org="Northwind Labs"):
    """Build the IR toolbox. action_log is a list that send_email/collection append to."""
    EV = Path(evidence_root)
    EV.mkdir(parents=True, exist_ok=True)

    # ---- knowledge (read-only over the plan + playbooks) ----
    def list_playbooks():
        """List the incident-response plan and playbooks, with their section titles."""
        out = []
        for doc, secs in knowledge.items():
            out.append(doc + ":")
            out += [f"    - {t}" for t, _ in secs]
        return "\n".join(out) if out else "(no playbooks loaded)"

    def search_playbooks(keyword):
        """Search the plan and playbooks for a keyword; returns [doc > section] matching lines."""
        key = (keyword or "").lower()
        out = []
        for doc, secs in knowledge.items():
            for title, body in secs:
                for line in body.splitlines():
                    if key in line.lower():
                        out.append(f"[{doc} > {title}] {line.strip()[:140]}")
                        if len(out) >= 25:
                            return "\n".join(out) + "\n...[more]"
        return "\n".join(out) if out else f"No playbook text matches {keyword!r}."

    def read_playbook(name, section=""):
        """Read a playbook (by file name) or one of its sections (by text in the title)."""
        want_doc = (name or "").lower()
        want_sec = (section or "").lower()
        hits = []
        for doc, secs in knowledge.items():
            if want_doc and want_doc not in doc.lower():
                continue
            for title, body in secs:
                if want_sec and want_sec not in title.lower():
                    continue
                hits.append(f"[{doc} > {title}]\n{body}")
        if not hits:
            return f"No playbook/section matching name={name!r} section={section!r}. Use list_playbooks."
        text = "\n\n".join(hits)
        return text[:2200] + ("" if len(text) <= 2200 else "\n...[truncated]")

    # ---- actions (human-in-the-loop) ----
    def draft_email(audience="management", incident=""):
        """Draft a notification email. Pulls a template from the playbooks if one exists.
        This DRAFTS only; it does not send. Show it to the team and ask for approval."""
        tmpl = None
        for doc, secs in knowledge.items():
            for title, body in secs:
                if "template" in title.lower() and (not incident or incident.lower() in doc.lower()
                                                     or incident.lower() in title.lower()):
                    tmpl = f"(template from [{doc} > {title}])\n{body}"
                    break
            if tmpl:
                break
        if tmpl:
            return tmpl
        return (f"To: {audience}\nSubject: Security incident in progress ({incident or 'under triage'})\n\n"
                f"Team, we are responding to a suspected {incident or 'security incident'} at {org}. "
                "The response team is engaged and following the IR plan. We will send an update within 60 "
                "minutes or on any material change. [No playbook template was found; this is a generic draft.]")

    def send_email(audience="management", approved=False, body=""):
        """Send the notification email - SIMULATED. Only call this AFTER the team approves in chat,
        with approved=True. Nothing real is sent; the action is written to the action log."""
        if not approved:
            return "Not sent. I need the team's explicit approval first (reply 'approve' to send)."
        entry = f"[DRILL/SIMULATED] Notification 'sent' to {audience}."
        action_log.append(entry)
        return entry + " (No real email was sent - this is a training drill.)"

    def recommend_collection(goal=""):
        """Recommend the exact tools, commands, and paths to collect evidence for `goal`,
        taken from the playbooks' collection sections."""
        key = (goal or "collect").lower()
        hits = []
        for doc, secs in knowledge.items():
            for title, body in secs:
                if any(w in title.lower() for w in ("collect", "evidence", "acquire", "preserve")):
                    if not goal or key in (title + " " + body).lower() or key in doc.lower():
                        hits.append(f"[{doc} > {title}]\n{body}")
        if not hits:
            return "No collection section matched. Try search_playbooks('collect') or read_playbook(section='collection')."
        text = "\n\n".join(hits)
        action_log.append(f"Recommended collection steps for: {goal or 'the incident'}.")
        return text[:2000] + ("" if len(text) <= 2000 else "\n...[truncated]")

    # ---- evidence (read-only over artifacts the team uploads/collects) ----
    def _safe(path):
        base = EV.resolve()
        path = (path or ".").strip().lstrip("/")
        t = base if path in ("", ".") else (EV / path).resolve()
        if t != base and base not in t.parents:
            raise ValueError("path is outside the evidence folder")
        return t

    def list_evidence(path="."):
        """List the artifacts the team has collected into the evidence folder."""
        t = _safe(path)
        if not t.exists():
            return "(no evidence collected yet)"
        if t.is_file():
            return f"{path} is a file. Use read_evidence or run_sql."
        items = [e + ("/" if (t / e).is_dir() else "") for e in sorted(os.listdir(t))]
        return "\n".join(items) if items else "(no evidence collected yet)"

    def read_evidence(path, max_chars=1600):
        """Read a collected text/log artifact (or get a hint for a database)."""
        t = _safe(path)
        if not t.exists() or t.is_dir():
            return f"No such artifact: {path}"
        if t.read_bytes()[:15] == b"SQLite format 3":
            import sqlite3
            con = sqlite3.connect(f"file:{t}?mode=ro", uri=True)
            tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
            con.close()
            return f"[{path} is a SQLite database. Tables: {tables}. Use run_sql with a SELECT.]"
        text = t.read_text(errors="replace")
        return text[:max_chars] + ("" if len(text) <= max_chars else "\n...[truncated]")

    def search_evidence(keyword, max_hits=25):
        """Search the collected artifacts for a keyword; returns artifact:line matches."""
        base = EV.resolve()
        key = (keyword or "").lower()
        out = []
        for p in sorted(base.rglob("*")):
            if not p.is_file() or p.suffix.lower() not in _TEXT_EXT:
                continue
            for i, line in enumerate(p.read_text(errors="replace").splitlines(), 1):
                if key in line.lower():
                    out.append(f"{p.relative_to(base)}:{i}: {line.strip()[:140]}")
                    if len(out) >= max_hits:
                        return "\n".join(out) + "\n...[more]"
        return "\n".join(out) if out else f"No evidence matches {keyword!r}."

    def run_sql(path, query):
        """Run ONE read-only SELECT against a collected SQLite artifact."""
        import sqlite3
        t = _safe(path)
        if not t.exists():
            return f"No such artifact: {path}"
        q = (query or "").strip().rstrip(";")
        if not q.lower().startswith(("select", "pragma")):
            return "Only read-only SELECT queries are allowed."
        try:
            con = sqlite3.connect(f"file:{t}?mode=ro", uri=True)
            cur = con.execute(q)
            cols = [c[0] for c in cur.description] if cur.description else []
            rows = cur.fetchmany(50)
            con.close()
        except Exception as e:
            return f"SQL error: {e}"
        if not rows:
            return "(no rows)"
        return "\n".join([" | ".join(cols)] + [" | ".join(str(c) for c in r) for r in rows])

    return {
        "list_playbooks": list_playbooks, "search_playbooks": search_playbooks,
        "read_playbook": read_playbook, "draft_email": draft_email, "send_email": send_email,
        "recommend_collection": recommend_collection, "list_evidence": list_evidence,
        "read_evidence": read_evidence, "search_evidence": search_evidence, "run_sql": run_sql,
    }


def build_ir_system_prompt(tools, org="Northwind Labs"):
    return (
        f"You are Sectra, the incident first responder for {org}. A response team has called you about a "
        "possible security incident. Your job is to calmly walk them through the response, ONE step at a "
        "time, using ONLY the company's incident-response plan and playbooks (via your tools). This is a "
        "training DRILL: any emails are simulated, never real.\n\n"
        "TOOLS (use exactly one per step):\n" + ak.tool_descriptions(tools) + "\n\n"
        "HOW TO REPLY: send ONE JSON object and nothing else.\n"
        "To use a tool:\n"
        '  {"thought": "<one short sentence>", "action": "search_playbooks", "action_input": {"keyword": "ransomware"}}\n'
        "To speak to the team (give a step, ask a question, or ask for approval):\n"
        '  {"thought": "<one short sentence>", "final_answer": "<what you tell the team>", "evidence_path": "<the playbook section you used>"}\n\n'
        "HOW TO RESPOND TO AN INCIDENT:\n"
        "1. Identify the matching playbook (search_playbooks / read_playbook) and tell the team the FIRST step.\n"
        "2. Management notification: draft_email to show the team the message, then ASK them to approve it. "
        "Only after they reply with approval may you call send_email with approved=true. Never send without approval.\n"
        "3. Evidence: use recommend_collection to give the exact tools, commands, and paths from the playbook.\n"
        "4. When the team has collected artifacts (list_evidence shows them), read_evidence / search_evidence / "
        "run_sql to analyse them and answer their questions.\n\n"
        "RULES: base every instruction on the plan/playbooks and cite the section in evidence_path; do only one "
        "thing per step; keep answers short and actionable; never invent commands or data; ask for approval "
        "before any notification. /no_think"
    )


def quick_agent(message, client, model, tools, history=None, **kw):
    prompt = build_ir_system_prompt(tools)
    return run_agent(message, client, model, tools, prompt, history=history, **kw)


In [ ]:
%%writefile ir_app.py
"""
ir_app.py - Sectra First Responder: a Gradio incident-response copilot.

The team chats with Sectra during an incident. Sectra uses the company's IR plan and
playbooks to walk them through the response, drafts the management notification (and only
"sends" it - simulated - after the team approves in chat), recommends the exact collection
commands, and reads back any artifacts the team uploads. Everything is a drill; no real email.

Config via environment (optional):
  PROVIDER / MODEL / OPENROUTER_API_KEY   (as in the other labs)
  KNOWLEDGE_DIR    folder of the plan + playbooks (.md). Default: knowledge
  ORG              company name shown to the agent. Default: Northwind Labs
"""
import os
import shutil
import tempfile
import gradio as gr
import irkit as ik

PROVIDER = os.environ.get("PROVIDER", "openrouter")
MODEL = os.environ.get("MODEL") or None
ORG = os.environ.get("ORG", "Northwind Labs")
KNOWLEDGE_DIR = os.environ.get("KNOWLEDGE_DIR", "knowledge")

_client = None
_model = None


def get_client():
    global _client, _model
    if _client is None:
        _client, _model = ik.connect_model(PROVIDER, MODEL)
    return _client, _model


def _new_state():
    try:
        kn = ik.load_knowledge(KNOWLEDGE_DIR)
    except Exception:
        kn = {}
    return {"knowledge": kn, "evidence_root": tempfile.mkdtemp(prefix="evidence_"), "action_log": []}


def _fmt(args):
    return ", ".join(f"{k}={v!r}" for k, v in (args or {}).items())


def _log_md(state):
    log = (state or {}).get("action_log", [])
    if not log:
        return "*No actions yet. Sectra will record notifications and collection steps here (all simulated).*"
    return "### Action log (drill)\n" + "\n".join(f"- {e}" for e in log)


def _as_text(c):
    """Gradio can round-trip message content as a list of parts; coerce to a string."""
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        return "".join((x.get("text") or x.get("content") or "") if isinstance(x, dict) else str(x)
                        for x in c)
    return str(c or "")


def _pairs(history):
    out, pending = [], None
    for m in history or []:
        if m.get("role") == "user":
            pending = _as_text(m.get("content"))
        elif m.get("role") == "assistant" and pending is not None:
            out.append((pending, _as_text(m.get("content")).split("---")[-1].strip()))
            pending = None
    return out[-5:]


def load_playbooks(files, state):
    if not files:
        return state, _playbook_status(state)
    d = tempfile.mkdtemp(prefix="playbooks_")
    for f in files:
        if f and f.lower().endswith(".md"):
            shutil.copy(f, d)
    try:
        state = {**state, "knowledge": ik.load_knowledge(d)}
    except Exception as e:
        return state, f"Could not read those playbooks: {e}"
    return state, _playbook_status(state)


def _playbook_status(state):
    kn = (state or {}).get("knowledge", {})
    return f"**Playbooks loaded:** {', '.join(kn) if kn else '(none)'}"


def add_evidence(files, state):
    if not files:
        return "No artifacts added."
    root = state["evidence_root"]
    names = []
    for f in files:
        if f:
            shutil.copy(f, root)
            names.append(os.path.basename(f))
    return f"**Collected into the evidence folder:** {', '.join(names)}"


def on_user(message, history):
    if not message.strip():
        return "", history
    return "", (history or []) + [{"role": "user", "content": message}]


def on_bot(history, state):
    q = _as_text(history[-1]["content"])
    try:
        client, model = get_client()
    except Exception as e:
        history.append({"role": "assistant", "content": f"Could not connect to the model: {e}"})
        yield history, _log_md(state)
        return
    tools = ik.make_ir_tools(state["knowledge"], state["evidence_root"], state["action_log"], org=ORG)
    prompt = ik.build_ir_system_prompt(tools, org=ORG)
    trace = "*Checking the plan and playbooks...*\n\n"
    history = history + [{"role": "assistant", "content": trace}]
    yield history, _log_md(state)
    final = None
    try:
        for ev in ik.run_agent_iter(q, client, model, tools, prompt, max_steps=14, history=_pairs(history[:-2])):
            if "final_answer" in ev:
                final = ev
            else:
                trace += f"{ev['step']}. **{ev['action']}**(`{_fmt(ev['args'])}`)\n\n"
                history[-1]["content"] = trace
                yield history, _log_md(state)
    except Exception as e:
        history[-1]["content"] = trace + f"\n\n**Model call failed:** {e}\n(Free models are rate-limited; try again.)"
        yield history, _log_md(state)
        return
    if final and final.get("final_answer"):
        cite = final.get("evidence_path")
        ans = final["final_answer"] + (f"\n\n`playbook: {cite}`" if cite else "")
    else:
        ans = "I could not find that in the plan or playbooks. Tell me the incident type, or ask about a specific step."
    history[-1]["content"] = trace + "\n---\n" + ans
    yield history, _log_md(state)


INTRO = (
    f"# Sectra First Responder\n"
    f"An incident-response copilot for **{ORG}** (training drill). Tell Sectra what you think is "
    "happening; it walks you through the response from the company's plan and playbooks, drafts the "
    "management notification (and only **sends it - simulated -** after you approve), recommends the "
    "exact collection commands, and reads the artifacts you upload.\n\n"
    f"*Model: `{PROVIDER}` / `{MODEL or 'default Qwen'}`  ·  All emails are simulated; nothing is really sent.*"
)

EXAMPLES = [
    "We think we have ransomware on a finance workstation.",
    "A user reported a phishing email and may have entered their password.",
    "We're seeing large unusual uploads from one host - possible data theft.",
    "Draft the management notification.",
    "Approve - send it.",
    "What evidence should we collect, and how?",
    "We uploaded the security log - what does it show?",
]


def build_demo():
    with gr.Blocks(title="Sectra First Responder") as demo:
        state = gr.State(_new_state())
        gr.Markdown(INTRO)
        with gr.Row():
            with gr.Column(scale=2):
                chat = gr.Chatbot(height=430, show_label=False)
                msg = gr.Textbox(placeholder="Tell Sectra what's happening, or reply (e.g. 'approve')...",
                                 show_label=False)
                gr.Examples(examples=EXAMPLES, inputs=msg)
            with gr.Column(scale=1):
                pb = gr.Markdown(_playbook_status(_new_state()))
                pb_up = gr.File(label="Replace playbooks (.md, optional)", file_count="multiple",
                                file_types=[".md"], type="filepath")
                ev_up = gr.File(label="Upload collected artifacts (logs, csv, db...)",
                                file_count="multiple", type="filepath")
                ev_status = gr.Markdown("*Collected artifacts appear here.*")
                action_log = gr.Markdown(_log_md(None))
        pb_up.change(load_playbooks, [pb_up, state], [state, pb])
        ev_up.change(add_evidence, [ev_up, state], [ev_status])
        msg.submit(on_user, [msg, chat], [msg, chat]).then(on_bot, [chat, state], [chat, action_log])
    return demo


def launch(**kwargs):
    opts = dict(theme=gr.themes.Soft())
    opts.update(kwargs)
    build_demo().launch(**opts)


if __name__ == "__main__":
    launch()


In [ ]:
%%writefile knowledge/00_IR_Plan.md
# Northwind Labs - Incident Response Plan (SYNTHETIC, for training)

## Purpose and scope
This plan governs how Northwind Labs responds to suspected security incidents on corporate
endpoints, servers, and cloud accounts. It is deliberately simple and is used for drills.

## Severity levels
- SEV-1 (Critical): active ransomware, confirmed data theft, or business-down. Notify management immediately.
- SEV-2 (High): credential compromise, malware on a single host, confirmed phishing with clicks.
- SEV-3 (Low): suspicious email reported, single blocked alert, policy violation.

## Roles
- Incident Commander (IC): owns the response and decisions.
- Communications Lead: handles management and, if needed, customer/legal messaging.
- Forensics Lead: collects and analyses evidence.
- IT On-call: performs containment actions (isolation, disabling accounts).

## Key contacts
- Management distribution list: management@northwind.example
- Legal: legal@northwind.example
- Security on-call: soc@northwind.example
- IT on-call: itoncall@northwind.example

## Response phases
1. Identify: confirm the incident and classify severity.
2. Contain: stop the spread (isolate hosts, disable accounts) without destroying evidence.
3. Eradicate: remove the attacker's foothold.
4. Recover: restore services from known-good state and monitor.
5. Notify: inform management (always for SEV-1/SEV-2); legal and customers per Legal's guidance.
6. Lessons learned: write up the timeline and improvements within 5 business days.

## When to escalate
- Escalate to SEV-1 and notify management immediately on: ransomware/encryption, confirmed
  exfiltration of customer data, or compromise of a privileged/admin account.

## Golden rules
- Preserve evidence before changing a system: do NOT power off a suspected ransomware host
  (you lose memory); isolate it from the network instead.
- Keep an action log with timestamps.
- Notify management before taking externally visible action.


In [ ]:
%%writefile knowledge/playbook_data_exfiltration.md
# Data Exfiltration Playbook (SYNTHETIC, for training)

## Classification
Suspected or confirmed theft of customer or sensitive data is SEV-1. Notify management and Legal
immediately.

## Indicators
- Large outbound transfers to unfamiliar hosts; spikes in DNS to a single rare domain (possible
  DNS tunnelling); cloud-storage uploads outside business use.
- Access to sensitive shares by an account that does not normally touch them.
- Security alerts for archiving tools (7-Zip, rar) running before an upload.

## Immediate response
1. Declare SEV-1; engage Legal (data-breach obligations may apply).
2. Identify the account and host involved; disable the account and isolate the host.
3. Preserve network and host evidence before blocking, so the scope can be measured.
4. Notify management and Legal using the template below.

## Management notification template
To: management@northwind.example
Subject: SEV-1 Security Incident - Suspected Data Exfiltration
Body:
Team, we are investigating a suspected exfiltration of sensitive data. The involved account has
been disabled and the host isolated. Evidence is being preserved so we can determine what, if
anything, left the environment. Legal has been engaged. We will provide an assessment within 2
hours; please avoid external statements until then.

## Evidence collection
- Proxy and firewall logs for the host (export E:\IR\exfil\proxy.csv, firewall.csv).
- DNS query logs for the host (look for high-volume or long subdomains - DNS tunnelling).
- EDR process tree and file-access history for the account.
- The destination domains/IPs and byte counts, to size the transfer.
Commands/examples:
- Review DNS: filter dns_queries.csv for the host and sort by query count.
- Netflow: sum bytes_out grouped by destination IP.

## Containment and recovery
- Block the destination domains/IPs; revoke the account's tokens and keys.
- Work with Legal on notification duties. Preserve everything; do not wipe until Legal clears it.


In [ ]:
%%writefile knowledge/playbook_phishing.md
# Phishing Playbook (SYNTHETIC, for training)

## Classification
A reported phishing email with no clicks is SEV-3. If a user entered credentials or ran an
attachment, treat it as SEV-2 and notify management.

## Indicators
- Reported suspicious email; mismatched sender/display name; look-alike domain.
- A user reports entering their password on a linked page, or opening an attachment.
- Mail gateway alerts; one user with an unusual sign-in location right after the email.

## Immediate response
1. Ask the user not to delete the email; have them forward it as an attachment to soc@northwind.example.
2. If credentials were entered: reset that user's password and revoke active sessions (IT on-call).
3. Search the mail system for other recipients of the same message and quarantine it.
4. If an attachment was run, isolate the host and treat as possible malware (see ransomware playbook).

## Management notification template
To: management@northwind.example
Subject: SEV-2 Security Incident - Phishing with possible credential compromise
Body:
Team, a phishing email reached staff and at least one user may have entered credentials. We have
reset the affected account, revoked its sessions, and are removing the message from other mailboxes.
No evidence of further access yet. We will update within 60 minutes.

## Evidence collection
- The reported email, saved as .eml (keep full headers).  Path: E:\IR\phish\message.eml
- Mail gateway logs for the sender/subject (export to E:\IR\phish\gateway.csv).
- Sign-in logs for the affected user (export the last 7 days).
- If a link was visited: proxy/DNS logs for the look-alike domain.

## Containment and recovery
- Block the sender domain and the phishing URL at the gateway and proxy.
- Enforce MFA re-registration for the affected user. Monitor the account for 7 days.


In [ ]:
%%writefile knowledge/playbook_ransomware.md
# Ransomware Playbook (SYNTHETIC, for training)

## Classification
Ransomware is a SEV-1 (Critical) incident. Notify management immediately (see template below).

## Indicators
- Files renamed with an unusual extension; ransom notes (e.g. README_RESTORE.txt) in folders.
- Mass file-modification alerts; shadow copies (VSS) deleted; backups failing.
- Security log 1102 (log cleared) or many 4688 process-creation events spawning encryptors.

## Immediate response (first 15 minutes)
1. Declare SEV-1 and assign an Incident Commander.
2. ISOLATE the affected host(s) from the network. Do NOT power off (memory is evidence).
3. Disable the compromised user/service accounts (IT on-call).
4. Notify management using the template in this playbook.

## Management notification template
To: management@northwind.example
Subject: SEV-1 Security Incident - Suspected Ransomware
Body:
Team, the security team has declared a SEV-1 incident: suspected ransomware affecting one or
more endpoints. Affected systems are being isolated now, and evidence is being preserved. We
have engaged the incident-response plan and will provide an update within 60 minutes or sooner
on any material change. No customer action is required at this time.

## Evidence collection
Collect, in this order, before eradication. Use an external, write-protected drive at E:\IR.
- Memory image (volatile - do first):   winpmem.exe -o E:\IR\mem.raw
- Windows event logs:                   wevtutil epl Security E:\IR\Security.evtx
                                         wevtutil epl System E:\IR\System.evtx
- NTFS artifacts:                        collect $MFT, $LogFile, and C:\Windows\Prefetch\*.pf
- Program execution:                     C:\Windows\appcompat\Programs\Amcache.hve
- Ransom note + a few encrypted samples (do not open macros).
Hash every artifact (sha256) and record it in the action log.

## Containment and eradication
- Block the malware's C2 domains/IPs at the firewall.
- Rebuild affected hosts from known-good images; restore data from offline backups only after
  confirming the backups are clean.

## Recovery
- Reset credentials for all accounts used on affected hosts. Monitor for re-infection for 14 days.


In [ ]:
%%writefile evidence/dns_queries.csv
timestamp,host,query,count
2024-07-28T09:30:00,10.4.2.19,www.microsoft.com,14
2024-07-28T09:31:00,10.4.2.19,outlook.office365.com,22
2024-07-28T09:35:00,10.4.2.19,northwind.example,9
2024-07-28T09:40:00,10.4.2.19,a7f3k9d2.x1y2.exfil-sync.net,318
2024-07-28T09:41:00,10.4.2.19,b2c8m4e1.x1y2.exfil-sync.net,297
2024-07-28T09:42:00,10.4.2.19,update.googleapis.com,6
2024-07-28T09:45:00,10.4.2.19,c9d1n7f3.x1y2.exfil-sync.net,264


In [ ]:
%%writefile evidence/security_events.log
# SYNTHETIC Windows Security log excerpt (fictional, for training)
# fields: time  EventID  description
2024-07-28T09:41:12  4625  Failed logon  account=jmalik  src=10.4.2.19  reason=bad_password
2024-07-28T09:41:18  4625  Failed logon  account=jmalik  src=10.4.2.19  reason=bad_password
2024-07-28T09:41:25  4625  Failed logon  account=jmalik  src=10.4.2.19  reason=bad_password
2024-07-28T09:41:31  4625  Failed logon  account=administrator  src=10.4.2.19  reason=bad_password
2024-07-28T09:42:03  4624  Successful logon  account=jmalik  src=10.4.2.19  logon_type=3
2024-07-28T09:43:50  4688  Process created  user=jmalik  image=C:\Windows\System32\cmd.exe  cmdline="cmd.exe /c vssadmin delete shadows /all /quiet"
2024-07-28T09:44:02  4688  Process created  user=jmalik  image=C:\Users\jmalik\Downloads\7z.exe  cmdline="7z a -p backup.7z C:\Finance\*"
2024-07-28T09:45:10  7045  Service installed  name=SysUpdateSvc  image=C:\Windows\Temp\su.exe  start=auto
2024-07-28T09:47:33  1102  The audit log was cleared  by=jmalik
2024-07-28T09:48:00  4688  Process created  user=jmalik  image=C:\Windows\Temp\su.exe  cmdline="su.exe -run"


### 3. Connect to a free Qwen model (OpenRouter)

Get a free key at **https://openrouter.ai/keys**, run this, and paste it (hidden).
*(Local Ollama instead? set `os.environ['PROVIDER']='ollama'` and skip the key.)*

In [ ]:
import os, getpass
os.environ['PROVIDER'] = os.environ.get('PROVIDER', 'openrouter')
if os.environ['PROVIDER'] == 'openrouter' and not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Paste your OpenRouter API key (hidden): ').strip()

### 4. Launch Sectra

`share=True` prints a public link in Colab. Try: *"We think we have ransomware"*, then reply
*"draft the notification"*, then *"approve"*, then upload the bundled `evidence/security_events.log`
and ask *"what does the log show?"*.

In [ ]:
import ir_app
ir_app.launch(share=True)